In [ ]:
# library
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
import time

from datasets import concatenate_datasets, load_dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

from google.colab import drive
drive.mount('/content/drive')

Torch: 2.10.0+cu128
CUDA available: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# downloaded package
!pip install -q unsloth
!pip install -q transformers datasets accelerate peft trl bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [ ]:
# functions part 1

MODEL_PATH = "/content/drive/MyDrive/lora_model_newstart2"
BASE_MODEL = "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_model,
    MODEL_PATH,
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:259: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/1.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
SYSTEM_PROMPT = (
    "You are an SVG generation model.\n"
    "Given a user prompt, generate exactly one valid SVG.\n"
    "Strict Requirements:\n"
    "- The SVG canvas must be 256x256 with viewBox=\"0 0 256 256\"\n"
    "- Use only valid SVG elements such as svg, g, path, rect, circle, ellipse, line, polyline, polygon\n"
    "- Do not include scripts, animation, foreignObject, event handlers, or external references\n"
    "- The SVG must be valid parseable XML\n"
    "- The total SVG length must be under 16000 characters\n"
    "- The number of path elements must not exceed 256\n"
    "- The output must start with <svg xmlns=\"http://www.w3.org/2000/svg\" width=\"256\" height=\"256\" viewBox=\"0 0 256 256\"> and end with </svg>\n"
    "Do not include any explanation, markdown, or extra text.\n"
    "Return only the SVG string."
)

STANDARD_HEADER = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def build_prompt(user_prompt: str) -> str:
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"Create an SVG for: {user_prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

def get_assistant_text(decoded: str) -> str:
    marker = "<|im_start|>assistant\n"
    start = decoded.find(marker)
    if start != -1:
        decoded = decoded[start + len(marker):]
    end_marker = "<|im_end|>"
    end = decoded.find(end_marker)
    if end != -1:
        decoded = decoded[:end]
    return decoded.strip()

def extract_svg_from_assistant(decoded: str) -> str:
    assistant_text = get_assistant_text(decoded)
    m = SVG_REGEX.search(assistant_text)
    return m.group(0).strip() if m else ""

def force_standard_svg_header(svg_text: str) -> str:
    if not svg_text:
        return ""
    return re.sub(r"^<svg\b[^>]*>", STANDARD_HEADER, svg_text, count=1, flags=re.IGNORECASE)

def is_valid_svg(svg_text: str) -> bool:
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False

def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

In [ ]:
# functions part 3

def extract_svg(text: str) -> str:
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def generate_svg(prompt, max_new_tokens=600):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Create an SVG for: {prompt}"},
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(input_text, return_tensors="pt", padding=True)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    gen_ids = outputs[0][input_len:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=False)

    svg = extract_svg(decoded)
    svg = force_standard_svg_header(svg)

    return decoded, svg

In [ ]:
def generate_svg_batch(prompts, max_new_tokens=512):
    messages_list = []
    for prompt in prompts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f'''Create exactly one SVG for "{prompt}".

Rules:
- Output only one <svg>...</svg>
- width must be 256
- height must be 256
- viewBox must be "0 0 256 256"
- do not use 24x24 icon format
- do not output explanation or markdown
'''},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        messages_list.append(text)

    inputs = tokenizer(
        messages_list,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded_list = []
    svg_list = []

    for i in range(len(prompts)):
        input_len_i = int(inputs["attention_mask"][i].sum().item())
        gen_ids = outputs[i][input_len_i:]
        decoded = tokenizer.decode(gen_ids, skip_special_tokens=False)
        svg = extract_svg(decoded)
        svg = force_standard_svg_header(svg)

        decoded_list.append(decoded)
        svg_list.append(svg)

    return decoded_list, svg_list

In [ ]:
import pandas as pd
import time

TEST_PATH = "/content/drive/MyDrive/dl-spring-2026-svg-generation/test.csv"
OUTPUT_PATH = "/content/drive/MyDrive/submission_0329_night.csv"

test_df = pd.read_csv(TEST_PATH)

batch_size = 16
rows = []
invalid_count = 0
t0 = time.time()

for start in range(0, len(test_df), batch_size):
    batch_df = test_df.iloc[start:start+batch_size]
    prompts = batch_df["prompt"].tolist()
    ids = batch_df["id"].tolist()

    decoded_list, svg_list = generate_svg_batch(prompts, max_new_tokens=512)

    print(f"batch prompts = {len(prompts)}, decoded = {len(decoded_list)}, svg = {len(svg_list)}")

    batch_invalid = 0
    batch_valid = 0

    for i in range(len(prompts)):
        sample_id = ids[i]
        prompt = prompts[i]
        svg = svg_list[i] if i < len(svg_list) else ""

        if not is_valid_svg(svg):
            batch_invalid += 1
            invalid_count += 1
            svg = fallback_svg(prompt)
        else:
            batch_valid += 1

        rows.append({
            "id": sample_id,
            "svg": svg
        })

    print(f"[Batch {start // batch_size}] valid: {batch_valid}, invalid: {batch_invalid}")
    print(f"Processed {min(start + batch_size, len(test_df))}/{len(test_df)}")

sub_df = pd.DataFrame(rows)
sub_df.to_csv(OUTPUT_PATH, index=False)

elapsed = (time.time() - t0) / 60
print("Done!")
print("Rows in output:", len(sub_df))
print("Invalid count:", invalid_count)
print(f"Time: {elapsed:.2f} min")
print(sub_df.head())


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


batch prompts = 16, decoded = 16, svg = 16
[Batch 0] valid: 1, invalid: 15
Processed 16/1000


KeyboardInterrupt: 